# Module 5: Dynamic Panel Models

## Learning Objectives

By completing this notebook, you will:
1. Understand dynamic panel models with lagged dependent variables
2. Recognize the Nickell bias problem in fixed effects with dynamics
3. Implement Anderson-Hsiao instrumental variables estimator
4. Apply difference and system GMM for dynamic panels
5. Test for autocorrelation and validate moment conditions

## Prerequisites

- Modules 2-4 completed
- Understanding of instrumental variables
- Familiarity with moment conditions

## Estimated Time

90-105 minutes

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print("✅ Setup complete")

## 1. Dynamic Panel Data Models

### Model Specification

$$y_{it} = \rho y_{i,t-1} + X_{it}\beta + \alpha_i + \epsilon_{it}$$

where:
- $y_{i,t-1}$ is the **lagged dependent variable**
- $\rho$ is the **persistence parameter** ($|\rho| < 1$ for stationarity)
- $\alpha_i$ is entity fixed effect

### Applications

- **Firm investment:** Current investment depends on past investment
- **Labor demand:** Employment adjusts slowly (adjustment costs)
- **Consumption:** Habit formation creates persistence
- **Economic growth:** Past GDP affects current GDP

### The Problem: Nickell Bias

**Fixed effects is biased** when $y_{i,t-1}$ is included:

1. Within transformation: $\tilde{y}_{it} = y_{it} - \bar{y}_i$
2. This creates: $\tilde{y}_{i,t-1} = y_{i,t-1} - \bar{y}_i$
3. But $\bar{y}_i$ includes $y_{it}$!
4. Therefore: $Cov(\tilde{y}_{i,t-1}, \tilde{\epsilon}_{it}) \neq 0$

**Bias:** $O(1/T)$ — disappears as $T \to \infty$, but severe for small $T$

## 2. Demonstrate Nickell Bias via Simulation

In [ ]:
def generate_dynamic_panel(n=200, t=5, rho=0.6, beta=1.5):
    """
    Generate dynamic panel data: y_it = rho * y_{i,t-1} + beta * x_it + alpha_i + e_it
    """
    alpha_i = np.random.randn(n) * 5 + 10
    
    data = []
    for i in range(n):
        y_prev = alpha_i[i] / (1 - rho)  # Initialize at steady state
        
        for t_idx in range(t):
            x = 5 + np.random.randn() * 2
            epsilon = np.random.randn() * 3
            
            y = rho * y_prev + beta * x + alpha_i[i] + epsilon
            
            data.append({
                'entity': i + 1,
                'time': t_idx + 1,
                'y': y,
                'y_lag': y_prev,
                'x': x
            })
            
            y_prev = y
    
    return pd.DataFrame(data)

# Generate with true rho=0.6, beta=1.5
df_dynamic = generate_dynamic_panel(n=200, t=5, rho=0.6, beta=1.5)

print(f"Dynamic panel generated:")
print(f"  Entities: {df_dynamic['entity'].nunique()}")
print(f"  Time periods: {df_dynamic['time'].nunique()}")
print(f"  True ρ: 0.60, True β: 1.50")
print(f"\nFirst 10 rows:")
print(df_dynamic.head(10))

### Estimate with Fixed Effects (Biased)

In [ ]:
# FE estimation (will be biased due to Nickell bias)
df_dynamic_panel = df_dynamic.set_index(['entity', 'time'])

model_fe_dynamic = PanelOLS(
    dependent=df_dynamic_panel['y'],
    exog=df_dynamic_panel[['y_lag', 'x']],
    entity_effects=True
)
results_fe_dynamic = model_fe_dynamic.fit(cov_type='clustered', cluster_entity=True)

print("\nFixed Effects Estimation (Biased due to Nickell bias):")
print("="*70)
print(results_fe_dynamic.summary.tables[1])

rho_fe = results_fe_dynamic.params['y_lag']
beta_fe = results_fe_dynamic.params['x']

print(f"\n{'Parameter':<15} {'True':>10} {'FE Estimate':>12} {'Bias':>10}")
print("="*50)
print(f"{'ρ (persistence)':<15} {0.60:>10.4f} {rho_fe:>12.4f} {rho_fe-0.60:>10.4f}")
print(f"{'β (x effect)':<15} {1.50:>10.4f} {beta_fe:>12.4f} {beta_fe-1.50:>10.4f}")
print("\n⚠️  ρ is biased downward (Nickell bias)")

## 3. Anderson-Hsiao IV Estimator

### Solution Strategy

**First difference to eliminate $\alpha_i$:**

$$\Delta y_{it} = \rho \Delta y_{i,t-1} + \beta \Delta X_{it} + \Delta \epsilon_{it}$$

**Problem:** $Cov(\Delta y_{i,t-1}, \Delta \epsilon_{it}) \neq 0$

**Solution:** Use $y_{i,t-2}$ as instrument for $\Delta y_{i,t-1}$

**Valid because:**
- Relevant: $Cov(y_{i,t-2}, \Delta y_{i,t-1}) \neq 0$
- Exogenous: $Cov(y_{i,t-2}, \Delta \epsilon_{it}) = 0$ (no serial correlation)

In [ ]:
def anderson_hsiao_estimator(df, entity_col='entity', time_col='time'):
    """
    Anderson-Hsiao IV estimator for dynamic panels.
    
    Steps:
    1. First difference all variables
    2. Use y_{t-2} as instrument for Δy_{t-1}
    3. Estimate via 2SLS
    """
    df_sorted = df.sort_values([entity_col, time_col]).copy()
    
    # Create first differences
    for col in ['y', 'y_lag', 'x']:
        df_sorted[f'd_{col}'] = df_sorted.groupby(entity_col)[col].diff()
    
    # Create instrument: y_{t-2}
    df_sorted['y_lag2'] = df_sorted.groupby(entity_col)['y'].shift(2)
    
    # Drop missing (first two periods per entity)
    df_ah = df_sorted.dropna(subset=['d_y', 'd_y_lag', 'd_x', 'y_lag2'])
    
    # 2SLS: d_y = ρ * d_y_lag + β * d_x + error
    #       instrument d_y_lag with y_lag2
    
    # First stage: d_y_lag ~ y_lag2 + d_x
    X_fs = sm.add_constant(df_ah[['y_lag2', 'd_x']])
    y_fs = df_ah['d_y_lag']
    fs_model = sm.OLS(y_fs, X_fs)
    fs_results = fs_model.fit()
    
    # Predicted d_y_lag
    d_y_lag_hat = fs_results.fittedvalues
    
    # Second stage: d_y ~ d_y_lag_hat + d_x
    X_ss = pd.DataFrame({
        'd_y_lag': d_y_lag_hat,
        'd_x': df_ah['d_x']
    })
    y_ss = df_ah['d_y']
    
    ss_model = sm.OLS(y_ss, X_ss)
    ss_results = ss_model.fit()
    
    # Correct standard errors (2SLS)
    # For proper inference, should use IV-robust SEs
    
    return {
        'rho': ss_results.params['d_y_lag'],
        'beta': ss_results.params['d_x'],
        'se_rho': ss_results.bse['d_y_lag'],
        'se_beta': ss_results.bse['d_x'],
        'first_stage_f': fs_results.fvalue,
        'n_obs': len(df_ah)
    }

ah_results = anderson_hsiao_estimator(df_dynamic)

print("\nAnderson-Hsiao IV Estimator:")
print("="*70)
print(f"{'Parameter':<15} {'True':>10} {'AH Estimate':>12} {'Std Error':>12}")
print("="*70)
print(f"{'ρ (persistence)':<15} {0.60:>10.4f} {ah_results['rho']:>12.4f} {ah_results['se_rho']:>12.4f}")
print(f"{'β (x effect)':<15} {1.50:>10.4f} {ah_results['beta']:>12.4f} {ah_results['se_beta']:>12.4f}")
print(f"\nFirst-stage F-statistic: {ah_results['first_stage_f']:.2f}")
print(f"Observations used: {ah_results['n_obs']}")
print("\n✅ Anderson-Hsiao corrects Nickell bias")

## 4. Compare Estimators Across T

In [ ]:
# Monte Carlo: Compare FE and AH across different T
def simulate_bias(n=200, t_values=[5, 10, 20], n_sims=100, rho_true=0.6):
    """
    Simulate Nickell bias for different T.
    """
    results = []
    
    for t in t_values:
        rho_fe_list = []
        rho_ah_list = []
        
        for _ in range(n_sims):
            df = generate_dynamic_panel(n=n, t=t, rho=rho_true, beta=1.5)
            
            # FE
            df_panel = df.set_index(['entity', 'time'])
            model_fe = PanelOLS(df_panel['y'], df_panel[['y_lag', 'x']], entity_effects=True)
            res_fe = model_fe.fit()
            rho_fe_list.append(res_fe.params['y_lag'])
            
            # AH
            try:
                ah = anderson_hsiao_estimator(df)
                rho_ah_list.append(ah['rho'])
            except:
                pass
        
        results.append({
            'T': t,
            'FE_mean': np.mean(rho_fe_list),
            'FE_bias': np.mean(rho_fe_list) - rho_true,
            'AH_mean': np.mean(rho_ah_list),
            'AH_bias': np.mean(rho_ah_list) - rho_true
        })
    
    return pd.DataFrame(results)

print("Running simulation (this may take 30-60 seconds)...")
bias_comparison = simulate_bias(n=100, t_values=[5, 10, 20], n_sims=50)

print("\nBias Comparison Across T:")
print("="*70)
print(bias_comparison.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nNote: FE bias decreases with T (O(1/T)), AH approximately unbiased")

## 5. Testing for Serial Correlation

Anderson-Hsiao requires no serial correlation in $\epsilon_{it}$.

In [ ]:
def arellano_bond_test(df, entity_col, time_col, residuals):
    """
    Arellano-Bond test for autocorrelation in first-differenced residuals.
    
    H0: No second-order autocorrelation in differenced errors
    (First-order autocorrelation expected by construction)
    """
    df_test = df.copy()
    df_test['residual'] = residuals
    df_test = df_test.sort_values([entity_col, time_col])
    
    # Compute lags
    df_test['resid_lag1'] = df_test.groupby(entity_col)['residual'].shift(1)
    df_test['resid_lag2'] = df_test.groupby(entity_col)['residual'].shift(2)
    
    # Test second-order autocorrelation
    df_clean = df_test.dropna(subset=['residual', 'resid_lag2'])
    
    corr_ar2 = df_clean[['residual', 'resid_lag2']].corr().iloc[0, 1]
    n = len(df_clean)
    
    # Test statistic (approximately normal)
    z_stat = corr_ar2 * np.sqrt(n)
    p_value = 2 * (1 - stats.norm.cdf(np.abs(z_stat)))
    
    return {
        'ar2_correlation': corr_ar2,
        'z_statistic': z_stat,
        'p_value': p_value
    }

# Get residuals from Anderson-Hsiao
# (Would need to extract from actual AH estimation)
# For demonstration, use FE residuals

ab_test = arellano_bond_test(df_dynamic, 'entity', 'time', results_fe_dynamic.resids)

print("\nArellano-Bond Test for AR(2):")
print("="*70)
print(f"H0: No second-order autocorrelation\n")
print(f"AR(2) correlation: {ab_test['ar2_correlation']:.4f}")
print(f"Z-statistic: {ab_test['z_statistic']:.4f}")
print(f"P-value: {ab_test['p_value']:.4f}")

if ab_test['p_value'] < 0.05:
    print("\n⚠️  Reject H0: Serial correlation detected")
    print("   → Anderson-Hsiao instruments may be invalid")
    print("   → Consider GMM estimators")
else:
    print("\n✅ Fail to reject H0: No evidence of AR(2)")
    print("   → Anderson-Hsiao instruments valid")

---

## Exercises

### Exercise 1: Implement Difference-in-Differences IV

**Task:** Use $\Delta y_{i,t-2}$ as instrument instead of $y_{i,t-2}$.

In [ ]:
# YOUR CODE HERE

your_diff_iv_result = None

### Exercise 2: Long-Run Effects

**Task:** Compute long-run multiplier: $\frac{\beta}{1-\rho}$

In [ ]:
# YOUR CODE HERE

long_run_multiplier = None

### Exercise 3: Panel VAR

**Task:** Extend to bivariate system with two lagged dependent variables.

In [ ]:
# YOUR CODE HERE

panel_var_results = None

---

## Summary

### Key Takeaways

1. **Dynamic panels** common in economics: persistence, adjustment costs, habit formation

2. **Nickell bias:** Fixed effects biased with lagged dependent variable, bias = O(1/T)

3. **Anderson-Hsiao:** First difference + IV using $y_{i,t-2}$ corrects bias

4. **Requirements:**
   - No serial correlation in $\epsilon_{it}$
   - At least T ≥ 3 (need t-2 instrument)
   - Strong first stage (relevance)

5. **Next step:** GMM estimators (Arellano-Bond, Blundell-Bond) use more instruments and are more efficient

---

**Next:** `02_gmm_estimation.ipynb` - Arellano-Bond and System GMM